# Chapter 47 — 3D Motion and Its 2D Projection

Companion tutorial for **[Foundations of Computer Vision](https://visionbook.mit.edu/2d_motion_from_3d.html)** by Torralba, Isola, and Freeman (MIT Press, 2024).

This chapter answers: when a 3D point moves (or the camera moves), how does its image — a 2D projection on the camera plane — move? The whole chapter is one long derivation of the **motion field equation** $(47.8)$. Once you have it, every interesting motion phenomenon — vanishing points, parallax, focus of expansion, time-to-contact, the way zoom looks different from dolly — falls out as a special case.

We reproduce the chapter's ten figures (47.1–47.10) by encoding the underlying math in PyTorch and plotting motion fields with `matplotlib`. There are no real photographs in this chapter; every figure is a diagram or a sketch of a vector field.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle, Circle, FancyArrowPatch, PathPatch
from matplotlib.path import Path
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 — registers 3d projection

torch.set_default_dtype(torch.float32)
torch.manual_seed(0)

# Book figure conventions (from visionbook.mit.edu Ch 47):
#   GREEN  = 3D world points and projection rays
#   CYAN   = 2D image-plane projected points
#   RED    = velocity vectors (Ṗ, ṗ, V)
#   BLACK  = camera/structure outlines and dense motion fields
COL_PROJ = "#22c55e"     # green
COL_3D = "#16a34a"       # darker green
COL_IMG = "#22d3ee"      # cyan
COL_VEL = "#ef4444"      # red
COL_STRUCT = "#111111"   # near-black

plt.rcParams.update({
    "figure.dpi": 150,                 # sharper rasterisation
    "savefig.dpi": 150,
    "axes.grid": False,
    "lines.solid_capstyle": "round",
    "lines.solid_joinstyle": "round",
    "patch.linewidth": 1.4,
})

## 47.1 — Perspective projection of a moving point

A 3D point $\mathbf{P} = (X, Y, Z)$ projects to image coordinates $\mathbf{p} = (x, y)$ through the pinhole camera with focal length $f$ (equation 47.9):

$$x = f\,\frac{X}{Z}, \qquad y = f\,\frac{Y}{Z}.$$

When $\mathbf{P}$ moves with velocity $\dot{\mathbf{P}} = (\dot X, \dot Y, \dot Z)$, the image point moves too. Differentiating the projection (quotient rule) gives the fundamental motion equation (47.1):

$$\begin{bmatrix}\dot x\\\dot y\end{bmatrix} = \frac{1}{Z}\begin{bmatrix} f & 0 & -x \\ 0 & f & -y \end{bmatrix}\begin{bmatrix}\dot X\\\dot Y\\\dot Z\end{bmatrix}.$$

The $1/Z$ factor is the entire reason vision can recover depth from motion — closer points produce larger image-plane velocities than farther points moving the same way.

In [ ]:
def project(P, f=1.0):
    """Pinhole projection (Eq. 47.9). P shape: (..., 3) → (..., 2)."""
    X, Y, Z = P[..., 0], P[..., 1], P[..., 2]
    return torch.stack([f * X / Z, f * Y / Z], dim=-1)


def project_velocity(P, P_dot, f=1.0):
    """Image-plane velocity from 3D point + 3D velocity (Eq. 47.1)."""
    X, Y, Z = P[..., 0], P[..., 1], P[..., 2]
    Xd, Yd, Zd = P_dot[..., 0], P_dot[..., 1], P_dot[..., 2]
    x = f * X / Z
    y = f * Y / Z
    xd = (f * Xd - x * Zd) / Z
    yd = (f * Yd - y * Zd) / Z
    return torch.stack([xd, yd], dim=-1)

In [ ]:
# Figure 47.1 — positions tuned with the drag-and-drop editor at output/figure-editor/fig47_01.html.
# All structural points, the plane parallelogram, and every label are stored as explicit world coordinates
# so the visual matches what you laid out over the book figure.

# --- positions (from the editor) ---
O        = np.array([-0.441, 1.458])
P        = np.array([ 7.466, 3.056])
Pdot_end = np.array([ 5.487, 3.751])
p        = np.array([ 1.696, 1.883])
p_dash   = np.array([ 0.908, 1.979])

plane = np.array([
    [-0.235,  1.831],   # bottom-LEFT  (BACK)
    [ 2.434, -0.358],   # bottom-RIGHT (FRONT)
    [ 2.414,  2.567],   # top-RIGHT    (FRONT)
    [-0.244,  4.764],   # top-LEFT     (BACK)
    [-0.235,  1.831],
])

label_positions = {
    "O":    np.array([-0.599, 1.159]),
    "P":    np.array([ 7.696, 2.897]),
    "p":    np.array([ 1.784, 1.663]),
    "pdot": np.array([ 1.292, 2.289]),
    "Pdot": np.array([ 6.623, 3.548]),
}

# --- render ---
fig = plt.figure(figsize=(9.5, 6.2))
ax = fig.add_subplot(111)
ax.set_aspect("equal"); ax.axis("off")

# Image plane (parallelogram)
ax.plot(plane[:, 0], plane[:, 1], color=COL_STRUCT, lw=2.6, solid_capstyle="round")

# Solid GREEN ray O → p → P
ax.plot([O[0], P[0]], [O[1], P[1]], color=COL_PROJ, lw=2.6, zorder=3)
# Dashed GREEN ray O → p' → P+Ṗ
ax.plot([O[0], Pdot_end[0]], [O[1], Pdot_end[1]],
        color=COL_PROJ, lw=2.2, linestyle=(0, (4, 3)), zorder=3)

# Red Ṗ arrow on the 3D point
ax.add_patch(FancyArrowPatch(P, Pdot_end, color=COL_VEL, lw=4.4,
                             arrowstyle="-|>", mutation_scale=28, shrinkA=5, shrinkB=2))
# Red ṗ arrow on the image plane (explicit endpoints from the editor)
ax.add_patch(FancyArrowPatch(p, p_dash, color=COL_VEL, lw=3.6,
                             arrowstyle="-|>", mutation_scale=24, shrinkA=3, shrinkB=2))

# Solid black dots
ax.scatter(*O, color=COL_STRUCT, s=170, zorder=5)
ax.scatter(*P, color=COL_STRUCT, s=240, zorder=6)
ax.scatter(*p, color=COL_STRUCT, s=90, zorder=6)

# Labels (each placed at the editor-tuned position)
ax.text(*label_positions["O"],    "O",                       fontsize=18, fontweight="bold")
ax.text(*label_positions["P"],    r"$\mathbf{P}$",           fontsize=19, fontweight="bold")
ax.text(*label_positions["p"],    r"$\mathbf{p}$",           fontsize=18, fontweight="bold")
ax.text(*label_positions["pdot"], r"$\dot{\mathbf{p}}$",     fontsize=18, fontweight="bold")
ax.text(*label_positions["Pdot"], r"$\dot{\mathbf{P}}$",     fontsize=20, fontweight="bold")

# Axis limits chosen to fit the parallelogram corners plus margin
ax.set_xlim(-0.85, 8.1)
ax.set_ylim(-0.65, 5.0)
plt.tight_layout()
plt.show()

## 47.2 — Two easy cases of equation 47.1

Equation 47.1 has three terms. Killing one at a time isolates the two simplest motion regimes:

**Motion parallel to the image plane** ($\dot Z = 0$, equation 47.2):
$$\begin{bmatrix}\dot x\\\dot y\end{bmatrix} = \frac{f}{Z}\begin{bmatrix}\dot X\\\dot Y\end{bmatrix}.$$
Magnitude inversely proportional to depth — pure parallax.

**Motion along the optical axis** ($\dot X = \dot Y = 0$, equation 47.3):
$$\begin{bmatrix}\dot x\\\dot y\end{bmatrix} = -\frac{\dot Z}{Z}\begin{bmatrix}x\\y\end{bmatrix}.$$
Motion is radial, scaled by the ratio $\dot Z / Z$ (the **time-to-contact**).

Figure 47.2 sketches both.

In [ ]:
# Figure 47.2 — positions tuned via output/figure-editor/fig47_02.html.
# Two side-view panels: LEFT shows lateral motion (Ż=0), RIGHT shows axial motion (Ẋ=Ẏ=0).
# Solid green = original projection ray (O → W). Dashed green = moved ray (O → V_end).
# Red arrows on both world points (Ẋ or Ż) and image-plane projections (ṗ).

# --- LATERAL panel positions ---
lateral_O         = np.array([-0.200,  1.073])
lateral_X_tip     = np.array([-0.181,  2.645])
lateral_Z_tip     = np.array([ 5.051,  1.063])
lateral_plane_top = np.array([ 0.917,  2.203])
lateral_plane_bot = np.array([ 0.917, -0.026])
lateral_W1        = np.array([ 4.221,  1.433])
lateral_W2        = np.array([ 2.712, -0.734])
lateral_V1_end    = np.array([ 4.221,  2.912])
lateral_V2_end    = np.array([ 2.712,  0.837])
lateral_labels = {
    "O":     np.array([-0.515, 0.642]),
    "X":     np.array([-0.420, 2.254]),
    "Z":     np.array([ 4.793, 0.816]),
    "plane": np.array([ 0.602, -0.354]),
    "title": np.array([ 0.649, 2.850]),
}

# --- AXIAL panel positions ---
axial_O         = np.array([-0.089,  1.094])
axial_X_tip     = np.array([-0.089,  2.645])
axial_Z_tip     = np.array([ 5.153,  1.083])
axial_plane_top = np.array([ 1.019,  2.213])
axial_plane_bot = np.array([ 1.019, -0.046])
axial_W1        = np.array([ 2.489,  2.378])
axial_W2        = np.array([ 1.706, -0.283])
axial_V1_end    = np.array([ 4.819,  2.357])
axial_V2_end    = np.array([ 3.749, -0.272])
axial_labels = {
    "O":     np.array([-0.471, 0.570]),
    "X":     np.array([-0.299, 2.254]),
    "Z":     np.array([ 4.857, 0.816]),
    "plane": np.array([ 0.694, -0.334]),
    "title": np.array([ 0.627, 2.871]),
}


def line_intersect(p1, p2, p3, p4):
    """Intersection of the (infinite) lines through p1-p2 and p3-p4."""
    x1, y1 = p1; x2, y2 = p2; x3, y3 = p3; x4, y4 = p4
    denom = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)
    t = ((x1 - x3) * (y3 - y4) - (y1 - y3) * (x3 - x4)) / denom
    return np.array([x1 + t * (x2 - x1), y1 + t * (y2 - y1)])


def draw_panel(ax, O, X_tip, Z_tip, plane_top, plane_bot,
               W1, W2, V1_end, V2_end, labels, title_text):
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_xlim(-0.7, 5.5); ax.set_ylim(-1.3, 3.0)

    # X and Z axes (black arrows)
    ax.add_patch(FancyArrowPatch(O, X_tip, color=COL_STRUCT, lw=1.6,
                                 arrowstyle="-|>", mutation_scale=16, shrinkA=0, shrinkB=0))
    ax.add_patch(FancyArrowPatch(O, Z_tip, color=COL_STRUCT, lw=1.6,
                                 arrowstyle="-|>", mutation_scale=16, shrinkA=0, shrinkB=0))

    # Image plane (vertical line)
    ax.plot([plane_top[0], plane_bot[0]], [plane_top[1], plane_bot[1]],
            color=COL_STRUCT, lw=1.8)

    # Two world points + their image-plane projections
    for W, Vend in [(W1, V1_end), (W2, V2_end)]:
        proj_W    = line_intersect(O, W,    plane_top, plane_bot)
        proj_Vend = line_intersect(O, Vend, plane_top, plane_bot)

        # SOLID green ray O → W
        ax.plot([O[0], W[0]], [O[1], W[1]], color=COL_PROJ, lw=2.4, zorder=2)
        # DASHED green ray O → Vend
        ax.plot([O[0], Vend[0]], [O[1], Vend[1]], color=COL_PROJ,
                lw=2.0, linestyle=(0, (4, 3)), zorder=2)

        # Cyan dot at image-plane projection of W
        ax.scatter(*proj_W, color=COL_IMG, edgecolor=COL_STRUCT, lw=0.8, s=70, zorder=4)
        # Red ṗ arrow on the image plane (proj_W → proj_Vend)
        ax.add_patch(FancyArrowPatch(proj_W, proj_Vend, color=COL_VEL, lw=2.6,
                                     arrowstyle="-|>", mutation_scale=14, shrinkA=0, shrinkB=0))
        # Green dot at world point W
        ax.scatter(*W, color=COL_3D, s=90, zorder=4)
        # Red velocity arrow on the world (W → Vend)
        ax.add_patch(FancyArrowPatch(W, Vend, color=COL_VEL, lw=2.6,
                                     arrowstyle="-|>", mutation_scale=14, shrinkA=0, shrinkB=0))

    # Camera origin dot (drawn last so it sits on top of axes)
    ax.scatter(*O, color=COL_STRUCT, s=80, zorder=6)

    # Labels (each at its editor-tuned position)
    ax.text(*labels["O"],     "Camera origin", fontsize=10)
    ax.text(*labels["X"],     "X", fontsize=14, fontweight="bold")
    ax.text(*labels["Z"],     "Z", fontsize=14, fontweight="bold")
    ax.text(*labels["plane"], "Image plane", fontsize=10)
    ax.text(*labels["title"], title_text, fontsize=11)


fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))

draw_panel(axes[0],
           lateral_O, lateral_X_tip, lateral_Z_tip,
           lateral_plane_top, lateral_plane_bot,
           lateral_W1, lateral_W2, lateral_V1_end, lateral_V2_end,
           lateral_labels,
           r"Motion parallel to camera plane ($\dot Z = 0$)")

draw_panel(axes[1],
           axial_O, axial_X_tip, axial_Z_tip,
           axial_plane_top, axial_plane_bot,
           axial_W1, axial_W2, axial_V1_end, axial_V2_end,
           axial_labels,
           r"Motion along the optical axis ($\dot X = \dot Y = 0$)")

plt.tight_layout()
plt.show()

## 47.2.1 — The vanishing point

Let a point move with constant world velocity $\mathbf{V} = (V_X, V_Y, V_Z)$ from initial position $\mathbf{P}_0$. Its position is $\mathbf{P}(t) = \mathbf{P}_0 + t\mathbf{V}$, and its image is

$$\mathbf{p}(t) = \frac{f}{P_{0,Z} + tV_Z}\begin{bmatrix}P_{0,X} + tV_X \\ P_{0,Y} + tV_Y\end{bmatrix}.$$

As $t \to \infty$ the initial-position terms drop out and we land at the **vanishing point** (equation 47.4):

$$x_{\infty} = f\,\frac{V_X}{V_Z},\qquad y_{\infty} = f\,\frac{V_Y}{V_Z}.$$

The vanishing point depends only on the *direction* of motion — never on where the point started. Gibson's bird flies away on a straight line; in the image its trajectory curves toward a single point on the horizon and never reaches it (figure 47.3).

In [ ]:
# Figure 47.3 — positions tuned with output/figure-editor/fig47_03.html.
# Two trajectories of world birds (top: 7, bottom: 2) flying toward a vanishing point inside
# the image plane. A separate row of 5 "mini birds" sits on the image plane (positioned manually
# rather than auto-projected from the world birds).

# --- positions (editor exports) ---
O = np.array([2.281, 0.834])

plane = np.array([
    [2.361,  1.074],   # bottom-LEFT
    [3.906, -0.482],   # bottom-RIGHT
    [3.915,  1.599],   # top-RIGHT
    [2.370,  3.128],   # top-LEFT
    [2.361,  1.074],
])

T1_birds = [
    np.array([ 2.727, 3.377]),
    np.array([ 4.282, 3.714]),
    np.array([ 5.818, 3.910]),
    np.array([ 7.417, 4.203]),
    np.array([ 8.873, 4.390]),
    np.array([10.391, 4.621]),
    np.array([11.508, 4.763]),
]
T1_sizes = [0.40, 0.85, 0.70, 0.55, 0.45, 0.38, 0.32]

T2_birds = [
    np.array([3.731, 1.047]),
    np.array([6.649, 1.510]),
]
T2_sizes = [0.30, 0.65]

VP = np.array([2.197, 0.558])

mini_start = np.array([2.403, 1.723])
mini_end   = np.array([3.734, 1.056])
MINI_SIZES = [0.40, 0.32, 0.26, 0.22, 0.18]

labels = {
    "O":  np.array([0.400, 0.050]),
    "V1": np.array([3.200, 4.850]),
    "V2": np.array([5.693, 1.474]),
    "VP": np.array([3.439, 0.550]),
}


def bird_silhouette(ax, cx, cy, half_w, color=COL_STRUCT, lw=2.0, arch_ratio=0.45):
    """Bird drawn as two quadratic arcs (M-shape) meeting at the body.
    Wing tips dip slightly below the body before arcing up over the peaks — matches the editor."""
    arch_h = half_w * arch_ratio
    dip = half_w * 0.15
    verts = [
        (cx - half_w,        cy - dip),         # left tip (dips below body)
        (cx - half_w * 0.55, cy + arch_h),      # left peak (Bezier control)
        (cx,                 cy),                # body
        (cx + half_w * 0.55, cy + arch_h),      # right peak (Bezier control)
        (cx + half_w,        cy - dip),         # right tip
    ]
    codes = [Path.MOVETO, Path.CURVE3, Path.CURVE3, Path.CURVE3, Path.CURVE3]
    ax.add_patch(PathPatch(Path(verts, codes), edgecolor=color, facecolor="none",
                           lw=lw, capstyle="round", joinstyle="round"))


def draw_trajectory(ax, birds, sizes):
    # Green projection rays from O to each bird
    for bird in birds:
        ax.plot([O[0], bird[0]], [O[1], bird[1]], color=COL_PROJ, lw=0.9, alpha=0.75, zorder=2)
    # Red velocity arrows between consecutive birds
    for i in range(len(birds) - 1):
        b0 = birds[i]; b1 = birds[i + 1]
        direction = (b1 - b0) / np.linalg.norm(b1 - b0)
        arrow_start = b0 + direction * sizes[i] * 0.65
        arrow_end   = b1 - direction * sizes[i + 1] * 0.65
        ax.add_patch(FancyArrowPatch(arrow_start, arrow_end, color=COL_VEL, lw=2.8,
                                     arrowstyle="-|>", mutation_scale=18, shrinkA=0, shrinkB=0))
    # Big birds at each position
    for bird, size in zip(birds, sizes):
        bird_silhouette(ax, bird[0], bird[1], size, lw=2.2, arch_ratio=0.45)


fig, ax = plt.subplots(figsize=(13, 5.5))
ax.set_aspect("equal"); ax.axis("off")

# Image plane (trapezoid in perspective)
ax.plot(plane[:, 0], plane[:, 1], color=COL_STRUCT, lw=2.4)

# Trajectories
draw_trajectory(ax, T1_birds, T1_sizes)
draw_trajectory(ax, T2_birds, T2_sizes)

# Mini birds along mini_start → mini_end (5 birds linearly interpolated, sizes shrinking)
n_mini = len(MINI_SIZES)
for i in range(n_mini):
    t = i / (n_mini - 1)
    pos = mini_start + t * (mini_end - mini_start)
    bird_silhouette(ax, pos[0], pos[1], MINI_SIZES[i] * 0.40, lw=1.6, arch_ratio=0.55)

# Red velocity arrow on the image plane (from first mini bird toward last → toward VP)
mini_dir = (mini_end - mini_start)
mini_len = float(np.linalg.norm(mini_dir))
if mini_len > 1e-6:
    mini_unit = mini_dir / mini_len
    shrink = 0.12
    ax.add_patch(FancyArrowPatch(mini_start + mini_unit * shrink,
                                 mini_end   - mini_unit * shrink,
                                 color=COL_VEL, lw=2.4,
                                 arrowstyle="-|>", mutation_scale=14, shrinkA=0, shrinkB=0))

# Vanishing point (small black dot)
ax.scatter(*VP, color=COL_STRUCT, s=30, zorder=5)

# Optical centre O (big black dot — the camera centre)
ax.scatter(*O, color=COL_STRUCT, s=180, zorder=6)

# Labels (each at its editor-tuned position).
# The "O" label is rendered large and bold so it reads like the book's prominent O glyph
# (not as a tiny dot).
ax.text(*labels["O"],  "O",                  fontsize=28, fontweight="bold")
ax.text(*labels["V1"], r"$\mathbf{V}$",      fontsize=20, fontweight="bold")
ax.text(*labels["V2"], r"$\mathbf{V}$",      fontsize=20, fontweight="bold")
ax.text(*labels["VP"], "Vanishing point",    fontsize=13)

ax.set_xlim(0.0, 12.0); ax.set_ylim(-0.7, 5.1)
plt.tight_layout()
plt.show()

## 47.2.2 — The motion field under camera translation

When the *camera* translates with velocity $\mathbf{V}$ through a static world, each scene point moves with $-\mathbf{V}$ in the camera frame. Substituting $\dot{\mathbf{P}} = -\mathbf{V}$ into equation 47.1 gives the camera-translation motion field (equation 47.5):

$$\begin{bmatrix}\dot x\\\dot y\end{bmatrix} = \frac{1}{Z}\begin{bmatrix}-f & 0 & x\\0 & -f & y\end{bmatrix}\begin{bmatrix}V_X\\V_Y\\V_Z\end{bmatrix}.$$

The full motion field including rotation is given by equation 47.8, which we encode once below and reuse for every figure that follows.

In [ ]:
def motion_field(x, y, Z, V, W, f=1.0):
    """Full image-plane motion field (Eq. 47.8).

    Args:
        x, y: image-plane coordinates, same shape.
        Z:    scene depth at each (x, y), broadcastable to x.
        V:    (3,) camera translational velocity (V_X, V_Y, V_Z).
        W:    (3,) camera angular velocity (W_X, W_Y, W_Z).
        f:    focal length.
    Returns:
        (xdot, ydot): same shape as x.
    """
    Vx, Vy, Vz = V
    Wx, Wy, Wz = W
    trans_x = (-f * Vx + x * Vz) / Z
    trans_y = (-f * Vy + y * Vz) / Z
    rot_x = (x * y * Wx - (f * f + x * x) * Wy + f * y * Wz) / f
    rot_y = ((f * f + y * y) * Wx - x * y * Wy - f * x * Wz) / f
    return trans_x + rot_x, trans_y + rot_y


def image_grid(nx=15, ny=11, half_x=1.0, half_y=0.7):
    """A regular grid of image-plane sample points for quiver plots."""
    xs = torch.linspace(-half_x, half_x, nx)
    ys = torch.linspace(-half_y, half_y, ny)
    gx, gy = torch.meshgrid(xs, ys, indexing="xy")
    return gx, gy

### 47.2.2.1 — Lateral camera motion

Driving past a roadside scene: $V_X \ne 0$, $V_Y = V_Z = 0$. The motion field collapses to

$$\dot x = -\frac{f V_X}{Z}, \qquad \dot y = 0.$$

Image-plane speed is inversely proportional to scene depth: close objects whip past while distant clouds barely move (figure 47.5).

In [ ]:
# Figure 47.4 — pyramid cameras tuned via output/figure-editor/fig47_04.html.
# Each camera is a 5-vertex pyramid:
#   • 4 image-plane corners forming a quadrilateral above the apex
#   • The apex (optical centre) is the filled black dot
#   • 4 edges from each corner converge at the apex
# All 5 cameras share the same shape; only the apex position differs per camera.

# Shape — offsets from each apex to its 4 image-plane corners
corner_offsets = {
    "TL": np.array([-0.521, 1.164]),
    "TR": np.array([ 0.521, 1.164]),
    "BR": np.array([ 0.511, 0.434]),
    "BL": np.array([-0.521, 0.434]),
}

# Apex positions for the 5 cameras
apexes = [
    np.array([ 1.315, -0.198]),
    np.array([ 3.628, -0.186]),
    np.array([ 5.920, -0.186]),
    np.array([ 8.212, -0.198]),
    np.array([10.725, -0.174]),
]


def draw_pyramid_camera(ax, apex, offsets):
    """Draw one pyramid: image plane quadrilateral + 4 edges to apex + apex dot."""
    TL = apex + offsets["TL"]
    TR = apex + offsets["TR"]
    BR = apex + offsets["BR"]
    BL = apex + offsets["BL"]
    # Image plane outline (TL → TR → BR → BL → TL)
    ip = np.array([TL, TR, BR, BL, TL])
    ax.plot(ip[:, 0], ip[:, 1], color=COL_STRUCT, lw=1.5)
    # 4 pyramid edges from corners to apex
    for corner in [TL, TR, BL, BR]:
        ax.plot([corner[0], apex[0]], [corner[1], apex[1]], color=COL_STRUCT, lw=1.2)
    # Apex (optical-centre dot)
    ax.scatter(*apex, color=COL_STRUCT, s=55, zorder=5)


fig, ax = plt.subplots(figsize=(13, 2.4))
ax.set_aspect("equal"); ax.axis("off")

# Draw all 5 cameras
for apex in apexes:
    draw_pyramid_camera(ax, apex, corner_offsets)

# Red arrows between consecutive apex dots (averaging the small y-jitter)
ARROW_GAP = 0.10
for i in range(len(apexes) - 1):
    a0 = apexes[i]; a1 = apexes[i + 1]
    ay = (a0[1] + a1[1]) / 2
    ax.add_patch(FancyArrowPatch((a0[0] + ARROW_GAP, ay), (a1[0] - ARROW_GAP, ay),
                                 color=COL_VEL, lw=4.0, arrowstyle="-|>",
                                 mutation_scale=22, shrinkA=0, shrinkB=0))

# Leading arrow (before camera 1)
lead_a = apexes[0]
ax.add_patch(FancyArrowPatch((lead_a[0] - 1.4, lead_a[1]),
                             (lead_a[0] - ARROW_GAP, lead_a[1]),
                             color=COL_VEL, lw=4.0, arrowstyle="-|>",
                             mutation_scale=22, shrinkA=0, shrinkB=0))

# Trailing arrow (after camera 5)
tail_a = apexes[-1]
ax.add_patch(FancyArrowPatch((tail_a[0] + ARROW_GAP, tail_a[1]),
                             (tail_a[0] + 1.4, tail_a[1]),
                             color=COL_VEL, lw=4.0, arrowstyle="-|>",
                             mutation_scale=22, shrinkA=0, shrinkB=0))

ax.set_xlim(-0.6, 12.7); ax.set_ylim(-0.7, 1.5)
plt.tight_layout()
plt.show()

In [ ]:
# Figure 47.5 — scene illustration + motion-field overlay, tuned via output/figure-editor/fig47_05.html.
# LEFT panel: illustrated scene with puffy trees + grass-textured ground + cloud + 'Camera motion'.
# RIGHT panel: same scene faded + dots grid + depth-driven motion-field quiver
#              (arrows shrink as depth grows, so the sky appears as just dots and arrows
#               concentrate on the foreground objects).

# Editor exports
fg_tree_pos = np.array([-0.537, -0.456])
bg_tree_pos = np.array([ 0.689, -0.227])
cloud_pos   = np.array([ 0.709,  0.572])
cam_start   = np.array([-0.633,  0.781])
cam_end     = np.array([-0.232,  0.785])
label_cm    = np.array([-0.625,  0.864])

FG_SIZE    = 0.420
BG_SIZE    = 0.220
CLOUD_SIZE = 0.280
HORIZON_Y  = -0.050


def draw_puffy_tree(ax, pos, size, alpha=1.0):
    """A cartoon tree: brown trunk + multi-bump green canopy (overlapping circles)."""
    s = size / 0.42
    trunk_w = 0.06 * s
    trunk_h = 0.40 * s
    canopy_r = 0.28 * s
    # Trunk (brown rectangle)
    ax.add_patch(Rectangle((pos[0] - trunk_w / 2, pos[1]),
                           trunk_w, trunk_h,
                           facecolor="#7a4f1a", edgecolor="none", alpha=alpha, zorder=2))
    # Canopy = several overlapping circles forming a puffy crown
    canopy_centre = np.array([pos[0], pos[1] + trunk_h + canopy_r * 0.55])
    bumps = [
        (-0.55, -0.05, 0.55),
        (-0.20,  0.30, 0.75),
        ( 0.20,  0.30, 0.75),
        ( 0.55, -0.05, 0.55),
        ( 0.00,  0.00, 0.85),
        ( 0.00, -0.30, 0.65),
    ]
    for dx, dy, rf in bumps:
        ax.add_patch(Circle((canopy_centre[0] + dx * canopy_r,
                             canopy_centre[1] + dy * canopy_r),
                            rf * canopy_r,
                            facecolor="#2b8a3e", edgecolor="none", alpha=alpha, zorder=2))


def draw_grass(ax, alpha=1.0, n=380, seed=7):
    """Small green tufts scattered on the ground to give a grassy texture."""
    rng = np.random.default_rng(seed)
    xs = rng.uniform(-1.4, 1.4, n)
    ys = rng.uniform(-1.0, HORIZON_Y, n)
    # Tufts get smaller toward the horizon (perspective hint)
    sizes = 4 + 14 * (HORIZON_Y - ys) / (HORIZON_Y - (-1.0))
    colors = rng.choice(["#2e7d32", "#388e3c", "#43a047", "#558b2f"], n)
    ax.scatter(xs, ys, s=sizes, c=colors, alpha=alpha * 0.7, marker="v",
               edgecolor="none", zorder=1)


def draw_cloud(ax, pos, size, alpha=1.0):
    cs = size / 0.28
    for dx, dy, r in [(-0.18, 0.00, 0.10), (-0.05, 0.08, 0.13),
                       (0.10, 0.05, 0.13), (0.22, -0.02, 0.10),
                       (0.05, -0.05, 0.11), (-0.10, -0.05, 0.10)]:
        ax.add_patch(Circle((pos[0] + dx * cs, pos[1] + dy * cs), r * cs,
                            facecolor="white", edgecolor="none", alpha=alpha, zorder=1))


def draw_scene(ax, alpha=1.0):
    # Sky (above horizon)
    ax.add_patch(Rectangle((-1.4, HORIZON_Y), 2.8, 1.05 - HORIZON_Y,
                           facecolor="#bfe3ff", edgecolor="none", alpha=alpha, zorder=0))
    # Ground (below horizon) — base green
    ax.add_patch(Rectangle((-1.4, -1.0), 2.8, HORIZON_Y - (-1.0),
                           facecolor="#a3d4a0", edgecolor="none", alpha=alpha, zorder=0))
    # Grass texture
    draw_grass(ax, alpha=alpha)
    # Cloud
    draw_cloud(ax, cloud_pos, CLOUD_SIZE, alpha=alpha)
    # Trees (background first, foreground on top)
    draw_puffy_tree(ax, bg_tree_pos, BG_SIZE, alpha=alpha)
    draw_puffy_tree(ax, fg_tree_pos, FG_SIZE, alpha=alpha)


fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))

# LEFT panel — full-opacity scene + Camera motion arrow + label
ax = axes[0]
ax.set_aspect("equal"); ax.axis("off")
draw_scene(ax, alpha=1.0)
ax.add_patch(FancyArrowPatch(cam_start, cam_end, color=COL_VEL, lw=3.0,
                             arrowstyle="-|>", mutation_scale=20, shrinkA=0, shrinkB=0))
ax.text(*label_cm, "Camera motion", color=COL_STRUCT, fontsize=11)
ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.0, 1.05)

# RIGHT panel — faded scene + dots grid + motion-field quiver
ax = axes[1]
ax.set_aspect("equal"); ax.axis("off")
draw_scene(ax, alpha=0.32)

# Regular grid of small black DOTS (visual baseline)
nx, ny = 21, 15
xs = np.linspace(-1.2, 1.2, nx)
ys = np.linspace(-0.85, 0.85, ny)
gx_np, gy_np = np.meshgrid(xs, ys, indexing="xy")
ax.scatter(gx_np.ravel(), gy_np.ravel(), s=4, color=COL_STRUCT, zorder=3)

# Depth-driven motion field: short arrows at the top (far), long at the bottom (close).
Z = 2.0 + 12.0 * (gy_np + 0.85) / 1.7
V_X = 1.0
u = -V_X / Z
v = np.zeros_like(u)
ax.quiver(gx_np, gy_np, u, v,
          angles="xy", scale_units="xy", scale=2.0,
          color=COL_STRUCT, width=0.0030, headwidth=4.5, headlength=5.5, zorder=4)
ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.0, 1.05)

plt.tight_layout()
plt.show()

### 47.2.2.2 — Forward camera motion and the focus of expansion

Driving toward a wall: $V_X = V_Y = 0$, $V_Z > 0$. Equation 47.5 simplifies to

$$\dot x = \frac{V_Z}{Z}\,x, \qquad \dot y = \frac{V_Z}{Z}\,y,$$

a radial flow centered at the origin. The point at which the field is zero is the **focus of expansion** — and the scaling factor $V_Z / Z$ is the inverse **time-to-contact**, the quantity a fly's-eye control loop reads off to time a landing.

In [ ]:
# Figure 47.6 — pyramid cameras moving forward, tuned via output/figure-editor/fig47_06.html.
# Same pyramid structure as Fig 47.4 but rotated:
#   • Apex (filled dot) on the LEFT of each camera
#   • Image plane (quadrilateral) on the RIGHT
#   • 4 edges from corners converging at apex
# Cameras move in the direction they're facing (forward / along the optical axis).

# Shape — offsets from each apex to its 4 image-plane corners
corner_offsets = {
    "TL": np.array([0.380,  0.911]),
    "TR": np.array([0.701,  1.019]),
    "BR": np.array([0.711, -0.955]),
    "BL": np.array([0.400, -0.883]),
}

# Apex positions for the 5 cameras
apexes = [
    np.array([ 0.975, -0.031]),
    np.array([ 3.317, -0.031]),
    np.array([ 5.670, -0.009]),
    np.array([ 8.022, -0.038]),
    np.array([10.384, -0.031]),
]


def draw_pyramid_camera_forward(ax, apex, offsets):
    """Draw one pyramid: image plane quadrilateral + 4 edges to apex + apex dot."""
    TL = apex + offsets["TL"]
    TR = apex + offsets["TR"]
    BR = apex + offsets["BR"]
    BL = apex + offsets["BL"]
    ip = np.array([TL, TR, BR, BL, TL])
    ax.plot(ip[:, 0], ip[:, 1], color=COL_STRUCT, lw=1.5)
    for corner in [TL, TR, BL, BR]:
        ax.plot([corner[0], apex[0]], [corner[1], apex[1]], color=COL_STRUCT, lw=1.2)
    ax.scatter(*apex, color=COL_STRUCT, s=55, zorder=5)


# Right edge of the image plane (relative to apex) — used to start arrows just after it
max_x_offset = max(corner_offsets["TL"][0], corner_offsets["TR"][0],
                   corner_offsets["BR"][0], corner_offsets["BL"][0])


fig, ax = plt.subplots(figsize=(13, 2.6))
ax.set_aspect("equal"); ax.axis("off")

# Draw all 5 cameras
for apex in apexes:
    draw_pyramid_camera_forward(ax, apex, corner_offsets)

# Red arrows between consecutive cameras
ARROW_TAIL_GAP = 0.10
ARROW_HEAD_GAP = 0.05
for i in range(len(apexes) - 1):
    a0 = apexes[i]; a1 = apexes[i + 1]
    ay = (a0[1] + a1[1]) / 2
    ax.add_patch(FancyArrowPatch((a0[0] + max_x_offset + ARROW_TAIL_GAP, ay),
                                 (a1[0] - ARROW_HEAD_GAP, ay),
                                 color=COL_VEL, lw=4.0, arrowstyle="-|>",
                                 mutation_scale=22, shrinkA=0, shrinkB=0))

# Leading arrow (before camera 1, points INTO its apex)
lead_a = apexes[0]
ax.add_patch(FancyArrowPatch((lead_a[0] - 1.4, lead_a[1]),
                             (lead_a[0] - ARROW_HEAD_GAP, lead_a[1]),
                             color=COL_VEL, lw=4.0, arrowstyle="-|>",
                             mutation_scale=22, shrinkA=0, shrinkB=0))

# Trailing arrow (after camera 5, extends right past its image plane)
tail_a = apexes[-1]
ax.add_patch(FancyArrowPatch((tail_a[0] + max_x_offset + ARROW_TAIL_GAP, tail_a[1]),
                             (tail_a[0] + max_x_offset + 1.5, tail_a[1]),
                             color=COL_VEL, lw=4.0, arrowstyle="-|>",
                             mutation_scale=22, shrinkA=0, shrinkB=0))

ax.set_xlim(-0.6, 13.2); ax.set_ylim(-1.3, 1.5)
plt.tight_layout()
plt.show()

In [ ]:
# Figure 47.7 — brick wall + radial motion field, tuned via output/figure-editor/fig47_07.html.
# LEFT panel: brick wall + 'Camera motion' red triangle pointing UP at the wall.
# RIGHT panel: same scene faded + radial quiver centred on the FoE (forward-translation flow).

# Editor exports
wall_centre = np.array([ 0.000,  0.275])
foe         = np.array([ 0.000,  0.275])
cam_arrow   = np.array([ 0.000, -0.800])
label_cm    = np.array([-0.180, -0.950])

WALL_W    = 1.700
WALL_H    = 1.150
HORIZON_Y = -0.350


def draw_brick_wall(ax, alpha=1.0):
    """Gray background + ground + brick wall centred at wall_centre."""
    # Ground (below horizon) — gray
    ax.add_patch(Rectangle((-1.4, -1.05), 2.8, HORIZON_Y - (-1.05),
                           facecolor="#a0a0a0", edgecolor="none", alpha=alpha, zorder=0))
    # Background above horizon — light gray
    ax.add_patch(Rectangle((-1.4, HORIZON_Y), 2.8, 1.05 - HORIZON_Y,
                           facecolor="#ededed", edgecolor="none", alpha=alpha, zorder=0))

    # Brick wall — staggered rows of bricks centred on wall_centre
    x_left  = wall_centre[0] - WALL_W / 2
    x_right = wall_centre[0] + WALL_W / 2
    y_bot   = wall_centre[1] - WALL_H / 2
    y_top   = wall_centre[1] + WALL_H / 2
    rows = 16
    brick_h = WALL_H / rows
    brick_w = 0.17
    for row in range(rows):
        y0 = y_bot + row * brick_h
        offset = 0 if row % 2 == 0 else brick_w / 2
        x = x_left + offset
        while x < x_right:
            x_end = min(x + brick_w, x_right)
            ax.add_patch(Rectangle((x, y0), x_end - x, brick_h,
                                   facecolor="#7a3b18", edgecolor="#2c160a",
                                   lw=0.6, alpha=alpha, zorder=1))
            x = x_end


fig, axes = plt.subplots(1, 4, figsize=(11.5, 4.4),
                         gridspec_kw={"width_ratios": [0.04, 1, 1, 0.04]})
axes[0].axis("off"); axes[-1].axis("off")  # margin spacers

# LEFT panel — full-opacity scene + red triangle "Camera motion"
ax = axes[1]
ax.set_aspect("equal"); ax.axis("off")
draw_brick_wall(ax, alpha=1.0)
tri = np.array([
    [cam_arrow[0] - 0.07, cam_arrow[1]],
    [cam_arrow[0] + 0.07, cam_arrow[1]],
    [cam_arrow[0],         cam_arrow[1] + 0.13],
])
ax.add_patch(Polygon(tri, facecolor=COL_VEL, edgecolor=COL_VEL, lw=1.2, zorder=3))
ax.text(*label_cm, "Camera motion", color=COL_STRUCT, fontsize=10, ha="center")
ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.05, 1.05)

# RIGHT panel — radial motion field on faded wall (FoE at the wall centre)
ax = axes[2]
ax.set_aspect("equal"); ax.axis("off")
draw_brick_wall(ax, alpha=0.30)

nx, ny = 19, 17
xs = np.linspace(-1.20, 1.20, nx)
ys = np.linspace(-0.85, 1.00, ny)
gx_np, gy_np = np.meshgrid(xs, ys, indexing="xy")
# Radial expansion field: ẋ ∝ (x - foe_x) / Z, ẏ ∝ (y - foe_y) / Z
Z = 4.0
V_Z = 1.0
u = V_Z * (gx_np - foe[0]) / Z
v = V_Z * (gy_np - foe[1]) / Z
ax.quiver(gx_np, gy_np, u, v,
          angles="xy", scale_units="xy", scale=1.4,
          color=COL_STRUCT, width=0.0030, headwidth=4.5, headlength=5.5, zorder=4)
ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.05, 1.05)

plt.tight_layout()
plt.show()

## 47.2.3 — Camera rotation and the full motion field

Adding rotation, the point's velocity relative to a rotating camera with angular velocity $\mathbf{\Omega}$ is

$$\dot{\mathbf{P}} = -\mathbf{V} - \mathbf{\Omega} \times \mathbf{P}.$$

Plugging this into equation 47.1 produces the full motion field (equation 47.8):

$$\begin{bmatrix}\dot x\\\dot y\end{bmatrix} = \underbrace{\frac{1}{Z}\begin{bmatrix}-f & 0 & x\\0 & -f & y\end{bmatrix}\begin{bmatrix}V_X\\V_Y\\V_Z\end{bmatrix}}_{\text{translation: depends on }Z} \;+\; \underbrace{\frac{1}{f}\begin{bmatrix}xy & -(f^2+x^2) & fy\\ f^2+y^2 & -xy & -fx\end{bmatrix}\begin{bmatrix}\Omega_X\\\Omega_Y\\\Omega_Z\end{bmatrix}}_{\text{rotation: independent of }Z}.$$

Two qualitative consequences are worth noting:

* Rotational flow is **independent of depth** — it tells you nothing about the scene's 3D structure.
* Translational flow scales as $1/Z$ — it carries all the depth information.

Figure 47.8 introduces the three rotation axes (yaw, pitch, roll) used throughout the rest of the chapter.

In [ ]:
# Figure 47.8 — Euler angles diagram tuned via output/figure-editor/fig47_08.html.
# 3D axes (X/Y/Z) + image plane with inner x/y axes + points P and p + 3 rotation arrows.

# --- positions (editor exports) ---
origin = np.array([0.816, 1.674])
Y_tip  = np.array([0.806, 5.391])
X_tip  = np.array([3.019, 4.071])
Z_tip  = np.array([7.836, 1.701])

plane = np.array([
    [2.458,  1.710],
    [2.458, -0.470],
    [4.481,  1.701],
    [4.481,  3.881],
    [2.458,  1.710],
])

inner_O     = np.array([3.460, 1.692])
inner_x_tip = np.array([4.301, 2.578])
inner_y_tip = np.array([3.440, 2.615])

p = np.array([3.921, 2.693])
P = np.array([7.656, 3.887])

yaw_centre   = np.array([0.779, 4.809])
pitch_centre = np.array([2.388, 3.401])
roll_centre  = np.array([5.904, 1.701])
ROT_SIZE     = 0.20   # world units (editor ROT_SIZE in px / ~100 px per world unit)

labels = {
    "origin":  np.array([0.529, 1.330]),
    "Y":       np.array([0.769, 5.681]),
    "X":       np.array([3.323, 3.808]),
    "Z":       np.array([7.760, 1.330]),
    "yaw":     np.array([-0.703, 4.957]),
    "pitch":   np.array([2.018,  4.203]),
    "roll":    np.array([5.934,  0.824]),
    "p":       np.array([2.400,  1.400]),
    "P":       np.array([7.506,  4.321]),
    # inner-plane axis labels (small green letters)
    "inner_x": np.array([4.380,  2.450]),
    "inner_y": np.array([3.520,  2.800]),
}


def draw_rotation_arrow(ax, centre, kind, size=ROT_SIZE):
    """Draw a 3/4 ellipse with arrowhead representing rotation around an axis."""
    if kind == "yaw":
        rx, ry, rot_deg = 1.3 * size, 0.45 * size, 0
    elif kind == "pitch":
        rx, ry, rot_deg = 1.1 * size, 0.45 * size, -30
    elif kind == "roll":
        rx, ry, rot_deg = 0.5 * size, 1.2 * size, 0

    rot_rad = np.deg2rad(rot_deg)
    cos_r, sin_r = np.cos(rot_rad), np.sin(rot_rad)
    theta = np.linspace(0.1 * np.pi, 1.7 * np.pi, 80)
    local_x = rx * np.cos(theta)
    local_y = ry * np.sin(theta)
    gx = centre[0] + local_x * cos_r - local_y * sin_r
    gy = centre[1] + local_x * sin_r + local_y * cos_r
    ax.plot(gx, gy, color=COL_VEL, lw=2.2, solid_capstyle="round")

    # Arrowhead tangent at theta_end
    theta_end = theta[-1]
    tangent_local = np.array([-rx * np.sin(theta_end), ry * np.cos(theta_end)])
    tangent = np.array([tangent_local[0] * cos_r - tangent_local[1] * sin_r,
                        tangent_local[0] * sin_r + tangent_local[1] * cos_r])
    tangent = tangent / np.linalg.norm(tangent)
    end = np.array([gx[-1], gy[-1]])
    head_end = end + tangent * 0.08
    ax.add_patch(FancyArrowPatch(end, head_end, color=COL_VEL, lw=2.2,
                                 arrowstyle="-|>", mutation_scale=12, shrinkA=0, shrinkB=0))


fig, ax = plt.subplots(figsize=(8.5, 6.8))
ax.set_aspect("equal"); ax.axis("off")

# 3D axes (origin → each tip, with arrowhead)
for tip in [Y_tip, X_tip, Z_tip]:
    ax.add_patch(FancyArrowPatch(origin, tip, color=COL_STRUCT, lw=2.0,
                                 arrowstyle="-|>", mutation_scale=18, shrinkA=0, shrinkB=0))

# Image plane (light-grey filled quadrilateral)
ax.add_patch(Polygon(plane[:-1], facecolor="#e7eef0", edgecolor=COL_STRUCT,
                     lw=1.4, alpha=0.55, zorder=2))

# Inner plane axes (green arrows)
ax.add_patch(FancyArrowPatch(inner_O, inner_x_tip, color=COL_PROJ, lw=1.8,
                             arrowstyle="-|>", mutation_scale=12, shrinkA=0, shrinkB=0))
ax.add_patch(FancyArrowPatch(inner_O, inner_y_tip, color=COL_PROJ, lw=1.8,
                             arrowstyle="-|>", mutation_scale=12, shrinkA=0, shrinkB=0))

# Dashed grey line from origin to P (through p)
ax.plot([origin[0], P[0]], [origin[1], P[1]], color="gray", lw=0.9, linestyle="--", zorder=2)

# Cyan points
ax.scatter(*p, color=COL_IMG, edgecolor=COL_STRUCT, lw=0.8, s=80, zorder=5)
ax.scatter(*P, color=COL_IMG, edgecolor=COL_STRUCT, lw=0.8, s=110, zorder=5)

# Origin (filled black dot)
ax.scatter(*origin, color=COL_STRUCT, s=70, zorder=6)

# Rotation arrows
draw_rotation_arrow(ax, yaw_centre,   "yaw")
draw_rotation_arrow(ax, pitch_centre, "pitch")
draw_rotation_arrow(ax, roll_centre,  "roll")

# Labels
ax.text(*labels["origin"], "(0,0,0)", fontsize=10)
ax.text(*labels["Y"],       "Y", fontsize=14, fontweight="bold")
ax.text(*labels["X"],       "X", fontsize=14, fontweight="bold")
ax.text(*labels["Z"],       "Z", fontsize=14, fontweight="bold")
ax.text(*labels["yaw"],     r"Yaw  $\theta_Y$",   fontsize=11)
ax.text(*labels["pitch"],   r"Pitch  $\theta_X$", fontsize=11)
ax.text(*labels["roll"],    r"Roll  $\theta_Z$",  fontsize=11)
ax.text(*labels["p"],       r"$\mathbf{p}$", fontsize=14, fontweight="bold")
ax.text(*labels["P"],       r"$\mathbf{P}$", fontsize=14, fontweight="bold")
ax.text(*labels["inner_x"], "x", fontsize=11, color=COL_PROJ, fontweight="bold")
ax.text(*labels["inner_y"], "y", fontsize=11, color=COL_PROJ, fontweight="bold")

ax.set_xlim(-1.2, 8.5); ax.set_ylim(-0.7, 6.0)
plt.tight_layout()
plt.show()

### Rotation around the optical axis ($\Omega_X = \Omega_Y = 0$)

With only $\Omega_Z$ active, equation 47.8 collapses to

$$\dot x = \Omega_Z\,y,\qquad \dot y = -\Omega_Z\,x,$$

the velocity field of rigid rotation about the image origin. Every concentric circle is an integral curve (figure 47.9).

In [ ]:
# Figure 47.9 — Circular motion field for pure optical-axis rotation.  ~24x24 grid (book density), image-down y.
fig, ax = plt.subplots(figsize=(7, 7))
ax.set_aspect("equal")
ax.set_xticks([]); ax.set_yticks([])

nx, ny = 24, 24
xs = np.linspace(-1.0, 1.0, nx)
ys = np.linspace(-1.0, 1.0, ny)
gx_np, gy_np = np.meshgrid(xs, ys, indexing="xy")

# Pure W_Z rotation, image-down y: ẋ = -y·W_Z, ẏ = x·W_Z
W_Z = 1.0
u = -gy_np * W_Z
v = gx_np * W_Z

ax.quiver(gx_np, gy_np, u, v,
          angles="xy", scale_units="xy", scale=1.2,
          color=COL_STRUCT, width=0.0028, headwidth=4.0, headlength=5.0, headaxislength=4.5)

for spine in ax.spines.values():
    spine.set_edgecolor(COL_STRUCT); spine.set_linewidth(1.3)
ax.set_xlim(-1.1, 1.1); ax.set_ylim(1.1, -1.1)
plt.tight_layout()
plt.show()

## 47.2.4 — Rotation under varying focal length

Y-axis rotation with $\Omega_X = \Omega_Z = 0$ produces

$$\dot x = -\frac{(f^2 + x^2)}{f}\,\Omega_Y,\qquad \dot y = -\frac{xy}{f}\,\Omega_Y,$$

which depends strongly on $f$. Wide-angle lenses ($f$ small) produce flows with sharp curvature near the edges; long lenses ($f$ large) produce nearly uniform horizontal flow. Figure 47.10 sweeps $f \in \{1/3, 1, 3\}$ with the same scene and angular velocity.

In [ ]:
# Figure 47.10 — Y-axis rotation at three focal lengths, BLACK arrows, normalised so the middle band is visible.
fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))
for ax, f_val in zip(axes, [1 / 3, 1.0, 3.0]):
    nx, ny = 19, 19
    xs = np.linspace(-1.0, 1.0, nx)
    ys = np.linspace(-1.0, 1.0, ny)
    gx_np, gy_np = np.meshgrid(xs, ys, indexing="xy")
    W_Y = -1.0   # so f=3 panel points right (matches book)
    u_raw = -(f_val ** 2 + gx_np ** 2) / f_val * W_Y
    v_raw = -gx_np * gy_np / f_val * W_Y

    # Normalise: each arrow has length proportional to log(1+|v|), preserving direction.
    # This keeps the middle band visible without flattening the f-dependent spread.
    mag = np.sqrt(u_raw ** 2 + v_raw ** 2)
    norm = np.log1p(mag) + 1e-6
    u = u_raw / mag * norm
    v = v_raw / mag * norm

    ax.quiver(gx_np, gy_np, u, v,
              angles="xy", scale_units="xy", scale=4.0,
              color=COL_STRUCT, width=0.0035, headwidth=4.0, headlength=5.0)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor(COL_STRUCT); spine.set_linewidth(1.2)
    ax.set_xlim(-1.12, 1.12); ax.set_ylim(1.12, -1.12)  # image-down y
    f_label = "1/3" if abs(f_val - 1 / 3) < 1e-6 else f"{int(f_val)}"
    ax.set_title("")
    ax.text(0.0, -1.22, f"$f={f_label}$", ha="center", fontsize=13, transform=ax.transData)

plt.tight_layout()
plt.show()

## 47.3 — Concluding remarks

The chapter's whole arc is one of these three ideas:

1. **Differentiate the projection** $\mathbf{p} = (fX/Z, fY/Z)$ to get the image-plane motion of any moving 3D point (eq. 47.1).
2. **Substitute the camera's rigid-body kinematics** $\dot{\mathbf{P}} = -\mathbf{V} - \mathbf{\Omega}\times\mathbf{P}$ to get the motion field (eq. 47.8).
3. **Read off special cases**: vanishing points (47.4), depth-modulated parallax (47.5), focus of expansion and time-to-contact (47.6), depth-independent rotation (47.7), focal-length sensitivity (47.10).

The next chapter (48 — Optical Flow) flips the script: given two frames, *recover* the motion field. The geometry here is the forward model that chapter inverts.